# 01 — Análise Exploratória (EDA)

**Issue #4** · Milestone M1

**Pré-requisito:** rodar `00_Setup_Colab.ipynb` na mesma sessão (define `CSV_PATH`) ou ter o CSV em `/content/data/raw/`.

**Checklist:** balanceamento CT/T · economia · mapas · AWP · buy tier · bomba · correlações · leakage (`time_left`)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

FIGURES = Path("/content/reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

try:
    csv_path = CSV_PATH  # noqa: F821 — definido no 00
except NameError:
    csv_path = Path("/content/data/raw/csgo_round_snapshots.csv")

if not Path(csv_path).exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {csv_path}\nRode o notebook 00_Setup_Colab primeiro."
    )

print("CSV:", csv_path)

In [ ]:
def buy_tier(money: float) -> str:
    if money < 2000:
        return "eco"
    if money < 4500:
        return "half"
    if money < 10000:
        return "force"
    return "full"


df = pd.read_csv(csv_path)
df["money_diff"] = df["ct_money"] - df["t_money"]
df["health_diff"] = df["ct_health"] - df["t_health"]
df["armor_diff"] = df["ct_armor"] - df["t_armor"]
df["players_alive_diff"] = df["ct_players_alive"] - df["t_players_alive"]
df["ct_richer"] = df["money_diff"] > 0
df["richer_wins"] = (
    ((df["round_winner"] == "CT") & df["ct_richer"])
    | ((df["round_winner"] == "T") & ~df["ct_richer"])
)
df["target"] = (df["round_winner"] == "T").astype(int)
df["ct_buy_tier"] = df["ct_money"].map(buy_tier)
df["t_buy_tier"] = df["t_money"].map(buy_tier)

print("Shape:", df.shape)
print("Nulos:", df.isna().sum().sum())
print("\nVencedor:")
print(df["round_winner"].value_counts())
print(f"\nTime com mais $ vence: {df['richer_wins'].mean():.1%}")
print(f"money_diff médio (CT−T): {df['money_diff'].mean():.0f} USD")

## 1. Balanceamento CT vs T

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["round_winner"].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax, hue=counts.index, legend=False)
ax.set_title("Distribuição do vencedor do round")
ax.set_xlabel("Lado vencedor")
ax.set_ylabel("Quantidade de snapshots")
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f"{v:,}", ha="center", fontsize=9)
plt.show()
fig.savefig(FIGURES / "01_distribuicao_vencedor.png")
plt.close(fig)

## 2. Economia (`ct_money`, `t_money`, `money_diff`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["ct_money"], label="CT", ax=axes[0], kde=True, stat="density", alpha=0.6)
sns.histplot(df["t_money"], label="T", ax=axes[0], kde=True, stat="density", alpha=0.6)
axes[0].set_title("Economia total por lado (USD)")
axes[0].legend()
sns.boxplot(data=df, x="round_winner", y="money_diff", hue="round_winner", ax=axes[1], legend=False)
axes[1].axhline(0, color="gray", ls="--", lw=0.8)
axes[1].set_title("Diferença de economia (CT − T) por vencedor")
plt.tight_layout()
plt.show()
fig.savefig(FIGURES / "02_economia.png")
plt.close(fig)

## 3. Mapas

In [ ]:
map_counts = df["map"].value_counts()
map_win = df.groupby("map")["target"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=map_counts.index, y=map_counts.values, ax=axes[0], hue=map_counts.index, legend=False)
axes[0].set_title("Snapshots por mapa")
axes[0].tick_params(axis="x", rotation=30)
sns.barplot(x=map_win.index, y=map_win.values, ax=axes[1], hue=map_win.index, legend=False)
axes[1].axhline(0.5, color="gray", ls="--", lw=0.8)
axes[1].set_title("Taxa de vitória T por mapa")
axes[1].set_ylabel("Proporção vitórias T")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()
fig.savefig(FIGURES / "03_mapas.png")
plt.close(fig)

## 4. Buy tier (eco / half / force / full)

In [ ]:
ct_rates = df.groupby("ct_buy_tier")["target"].apply(lambda s: 1 - s.mean())
t_rates = df.groupby("t_buy_tier")["target"].mean()
order = ["eco", "half", "force", "full"]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(order))
width = 0.35
ax.bar(x - width / 2, [ct_rates.get(t, 0) for t in order], width, label="CT vence")
ax.bar(x + width / 2, [t_rates.get(t, 0) for t in order], width, label="T vence")
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_title("Taxa de vitória por tier de compra")
ax.set_ylabel("Proporção de vitórias")
ax.legend()
plt.show()
fig.savefig(FIGURES / "04_buy_tier.png")
plt.close(fig)

## 5. Armamento e AWP

In [ ]:
weapons = [c for c in df.columns if c.startswith(("ct_weapon_", "t_weapon_"))]
ct_weapons = [c for c in weapons if c.startswith("ct_weapon_")]
ct_totals = df[ct_weapons].sum().sort_values(ascending=False).head(10)
t_weapons = [c.replace("ct_", "t_") for c in ct_totals.index]
t_totals = df[t_weapons].sum().reindex(t_weapons)
labels = [c.replace("ct_weapon_", "").upper() for c in ct_totals.index]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width / 2, ct_totals.values, width, label="CT")
ax.bar(x + width / 2, t_totals.values, width, label="T")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.set_title("Top 10 armas nos snapshots")
ax.set_ylabel("Soma de contagem")
ax.legend()
plt.show()
fig.savefig(FIGURES / "05_armas_top10.png")
plt.close(fig)

awp_ct = df.groupby(df["ct_weapon_awp"] > 0)["target"].apply(lambda s: 1 - s.mean())
awp_t = df.groupby(df["t_weapon_awp"] > 0)["target"].mean()
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(
    x=["CT sem AWP", "CT com AWP", "T sem AWP", "T com AWP"],
    y=[awp_ct.get(False, np.nan), awp_ct.get(True, np.nan), awp_t.get(False, np.nan), awp_t.get(True, np.nan)],
    hue=["CT sem AWP", "CT com AWP", "T sem AWP", "T com AWP"],
    ax=ax,
    legend=False,
)
ax.set_title("Taxa de vitória T vs presença de AWP")
ax.set_ylabel("Proporção vitórias T")
ax.tick_params(axis="x", rotation=25)
plt.show()
fig.savefig(FIGURES / "06_awp_impacto.png")
plt.close(fig)

## 6. Bomba plantada

In [ ]:
rates = df.groupby("bomb_planted")["target"].mean()
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(
    x=["Não plantada", "Plantada"],
    y=[rates.get(False, 0), rates.get(True, 0)],
    hue=["Não plantada", "Plantada"],
    ax=ax,
    legend=False,
)
ax.axhline(0.5, color="gray", ls="--", lw=0.8)
ax.set_title("Taxa de vitória T vs bomba plantada")
ax.set_ylabel("Proporção vitórias T")
plt.show()
fig.savefig(FIGURES / "07_bomba_plantada.png")
plt.close(fig)

## 7. Tempo restante e jogadores vivos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["time_left"], bins=40, ax=axes[0], kde=True)
axes[0].set_title("Tempo restante no round (segundos)")
sample = df.sample(min(8000, len(df)), random_state=42)
sns.scatterplot(
    data=sample,
    x="players_alive_diff",
    y="money_diff",
    hue="round_winner",
    alpha=0.35,
    ax=axes[1],
)
axes[1].set_title("Jogadores vivos (CT−T) vs economia (amostra)")
plt.tight_layout()
plt.show()
fig.savefig(FIGURES / "08_tempo_jogadores_vivos.png")
plt.close(fig)

## 8. Correlações

In [ ]:
cols = [
    "time_left",
    "money_diff",
    "health_diff",
    "armor_diff",
    "players_alive_diff",
    "bomb_planted",
    "target",
]
corr = df[cols].copy()
corr["bomb_planted"] = corr["bomb_planted"].astype(int)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlação — variáveis-chave")
plt.show()
fig.savefig(FIGURES / "09_correlacao.png")
plt.close(fig)

### Leakage temporal (`time_left`)

O dataset contém **vários snapshots por round** em momentos diferentes. Variáveis como `time_left`, `health_diff` e `players_alive_diff` mudam **durante** o round e podem refletir o desfecho já em curso — isso é **data leakage** se usarmos snapshots do fim do round para prever o vencedor.

**Decisão do projeto (issue #5):** filtrar `time_left >= 150` para manter apenas snapshots próximos ao **início** do round, quando economia e loadout são mais preditivos e menos contaminados pelo andamento do combate.

## 9. Resumo numérico

In [ ]:
summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "nulls_total": int(df.isna().sum().sum()),
    "winner_counts": df["round_winner"].value_counts().to_dict(),
    "winner_pct_t": round(df["target"].mean() * 100, 2),
    "richer_wins_pct": round(df["richer_wins"].mean() * 100, 2),
    "maps": df["map"].value_counts().to_dict(),
    "time_left_mean": round(df["time_left"].mean(), 2),
    "time_left_median": round(df["time_left"].median(), 2),
    "money_diff_mean": round(df["money_diff"].mean(), 2),
    "bomb_planted_pct": round(df["bomb_planted"].mean() * 100, 2),
    "awp_ct_pct": round((df["ct_weapon_awp"] > 0).mean() * 100, 2),
    "awp_t_pct": round((df["t_weapon_awp"] > 0).mean() * 100, 2),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
(FIGURES / "eda_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"\nGráficos salvos em: {FIGURES}")

---
Próximo passo: **`02_Preprocessamento.ipynb`** (issues #5 e #6)